# 05 – Análise de erros e equidade

Previsões fora da amostra (validação + teste) do melhor candidato. Execute antes o notebook 03.

In [1]:
from pathlib import Path
import sys
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))
import pandas as pd
pd.set_option("display.width", 160, "display.max_columns", 30)
from src import config
from src.executar import executar, ler_resultados
R = ler_resultados()
print(R.get("aviso") or "Dados reais.")

Dados reais.


## Em que faixa do escore os erros se concentram

In [2]:
display(R["erro_faixa"].round(3))

,faixa do previsto,n,MAE,WAPE,viés,exames subestimados,exames superestimados
0,"(367.0, 666.0]",8.0,64.745,0.121,10.406,217.356,300.601
1,"(6.0, 66.0]",8.0,5.940,0.540,-3.638,38.311,9.207
2,"(66.0, 367.0]",8.0,36.024,0.195,7.131,115.573,172.621


## Quais unidades, municípios e meses são mal previstos

In [3]:
display(R["erro_unidade"].round(3))
if "erro_municipio" in R:
    display(R["erro_municipio"].round(3))
display(R["erro_mes"].round(3))

,unidade,n,MAE,WAPE,viés,exames subestimados,exames superestimados
0,Particulares,4.0,6.723,0.560,-4.431,22.307,4.584
1,Viver Mais,4.0,5.157,0.516,-2.845,16.004,4.623
2,São Lucas,4.0,24.775,0.269,2.866,43.819,55.282
3,Roseane Dore,4.0,47.273,0.170,11.396,71.754,117.339
4,SUS,4.0,74.219,0.135,73.870,0.697,296.178
5,Maurílio de Almeida,4.0,55.271,0.106,-53.059,216.659,4.423


,mes,n,MAE,WAPE,viés,exames subestimados,exames superestimados
0,2026-06-01,6.0,46.016,0.221,46.016,0.00,276.096
1,2026-08-01,6.0,39.769,0.171,13.709,78.18,160.435
2,2026-07-01,6.0,41.273,0.144,-41.273,247.64,0.000
3,2026-09-01,6.0,15.220,0.061,0.080,45.42,45.898


## Casos problemáticos e problemas de dado

In [4]:
display(R["casos_problematicos"].round(3))
print(R["dados"])

,unidade,mes,exames,previsto,erro,lag_1,variação real vs mês anterior,indício
0,SUS,2026-06-01,491,665.886,174.886,670.0,-0.267,sem salto: erro do modelo
1,Maurílio de Almeida,2026-07-01,593,467.377,-125.623,462.0,0.284,sem salto: erro do modelo
2,SUS,2026-08-01,510,606.806,96.806,625.0,-0.184,sem salto: erro do modelo
3,Roseane Dore,2026-07-01,344,272.246,-71.754,218.0,0.578,salto brusco no real: checar registro/retifica...
4,Roseane Dore,2026-06-01,218,286.831,68.831,271.0,-0.196,sem salto: erro do modelo
5,Maurílio de Almeida,2026-08-01,543,481.358,-61.642,593.0,-0.084,sem salto: erro do modelo
6,São Lucas,2026-07-01,132,95.129,-36.871,72.0,0.833,salto brusco no real: checar registro/retifica...
7,São Lucas,2026-08-01,62,94.693,32.693,132.0,-0.530,salto brusco no real: checar registro/retifica...


{'celulas_unidade_mes': 84, 'celulas_sem_registro (NaN)': 0, 'linhas_descartadas_sem_historico_ou_alvo': 18, 'unidades_com_algum_mes_ausente': 0, 'tem_municipio': False, 'n_unidades': 6, 'n_meses': 14}


## Equidade entre grupos
Diferença de WAPE entre municípios, portes e unidades. A diferença é reportada mesmo quando não puder ser corrigida.

In [5]:
display(R["equidade"].round(3))
for k, v in R["equidade_resumo"].items():
    print(k, "->", v)

,critério,grupo,n,WAPE,MAE,viés,viés relativo,amostra pequena
0,porte da unidade,grande,8,0.121,64.745,10.406,0.019,False
1,porte da unidade,média,8,0.195,36.024,7.131,0.039,False
2,porte da unidade,pequena,8,0.540,5.940,-3.638,-0.331,False
3,unidade,Maurílio de Almeida,4,0.106,55.271,-53.059,-0.101,False
4,unidade,Particulares,4,0.560,6.723,-4.431,-0.369,False
5,unidade,Roseane Dore,4,0.170,47.273,11.396,0.041,False
6,unidade,SUS,4,0.135,74.219,73.870,0.135,False
7,unidade,São Lucas,4,0.269,24.775,2.866,0.031,False
8,unidade,Viver Mais,4,0.516,5.157,-2.845,-0.285,False


porte da unidade -> {'pior_grupo': 'pequena', 'melhor_grupo': 'grande', 'wape_pior': 0.5399817214548953, 'wape_melhor': 0.12084846355360829, 'razao_wape': 4.468254751251842, 'diferenca_pontos_percentuais': 41.9133257901287, 'sinalizado': True, 'conclusao': 'diferença acima do limite combinado [H]: reportar e discutir com o demandante'}
unidade -> {'pior_grupo': 'Particulares', 'melhor_grupo': 'Maurílio de Almeida', 'wape_pior': 0.5602278815857646, 'wape_melhor': 0.10567988808249314, 'razao_wape': 5.301177847089068, 'diferenca_pontos_percentuais': 45.45479935032715, 'sinalizado': True, 'conclusao': 'diferença acima do limite combinado [H]: reportar e discutir com o demandante'}


## Variáveis proxy
Se `unidade` e `município` (que podem refletir perfil de público ou convênio) saem do modelo, o erro muda?

In [6]:
print(R["ablação_identificadores"])

{'modelo': 'Random forest', 'WAPE_com': 0.1762827509664446, 'WAPE_sem_unidade_e_municipio': 0.19041674012811677}
